# Week 7 — The AI as a Reader, at Scale  (completion problem — **fuller guidance** version)

**What you'll do today.** Use a big AI model (Gemini) to *judge* every item in a slice of your
corpus — the way a human coder would, but in minutes. You'll write a labeling prompt (the prompt
*is* your codebook), ask the model **how sure** it is, sort by confidence, trust the sure labels,
and **hand-check the unsure ones** against a small gold set.

> This notebook runs end-to-end **with no API key** using a recorded sample response, so you can
> read the whole pipeline first. For live calls, add a free Gemini key (below).
> Errors? `../kits/common-errors-cheatsheet.md`.

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q google-generativeai pandas

In [ ]:
import os, json
import pandas as pd
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"

# Read the key from a Colab secret / environment variable — NEVER hard-code it.
API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata  # only exists in Colab
    API_KEY = API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass
LIVE = bool(API_KEY) and not SMOKE
print("Live Gemini calls:", LIVE, "(False = using the recorded sample response)")

## Your corpus slice and your codebook (the prompt)

Here we classify short review-style sentences as `positive` / `negative` / `mixed`. Swap in your
own items and your own categories. **The prompt is the codebook**: tighten the definition, add an
example, and the labels shift — the same subjective call as stemming (W2) and weights (W3), now
in plain English.

In [ ]:
items = [
    "an absolute triumph, I cried twice and laughed more",
    "technically fine but I felt nothing the whole time",
    "a boring slog, I checked my phone constantly",
    "gorgeous to look at, hollow underneath",
    "best thing I have seen all year, full stop",
]
PROMPT = """You are labeling short opinions about films.
Return ONLY JSON: a list of objects with keys "label" and "confidence".
"label" is one of: positive, negative, mixed.
"confidence" is a number from 0 to 1 (how sure you are).
Label each item in order. Items:
{items}"""

## The recorded sample response (the cassette)

So the notebook runs without a key or network, we commit one recorded response. With a real key,
the live call replaces it. We assert on the *parsing and sorting logic*, not the model's exact
words — that's what makes this testable in CI.

In [ ]:
CASSETTE = json.dumps([
    {"label": "positive", "confidence": 0.98},
    {"label": "mixed",    "confidence": 0.55},
    {"label": "negative", "confidence": 0.95},
    {"label": "mixed",    "confidence": 0.60},
    {"label": "positive", "confidence": 0.99},
])

## Run the annotator

In [ ]:
def annotate(items):
    prompt = PROMPT.format(items="\n".join(f"- {x}" for x in items))
    if LIVE:
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        model = genai.GenerativeModel("gemini-2.5-flash")
        raw = model.generate_content(prompt).text
    else:
        raw = CASSETTE
    # TODO: the model sometimes wraps JSON in ```code fences```. Strip them, then parse.
    # Hint: .strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    raw = ...
    return json.loads(raw)

labels = annotate(items)
df = pd.DataFrame(labels); df["text"] = items
print(df.to_string(index=False))

## Confidence — sort, trust, and hand-check

Trust the labels the model is sure of; pull the **unsure** ones for hand-checking. (Those are
exactly the 30 to hand-label in the Week 6 calibration set — not a random 30.)

In [ ]:
df_sorted = df.sort_values("confidence")
print("Hand-check these (lowest confidence first):")
print(df_sorted[["confidence","label","text"]].head(3).to_string(index=False))

## Accuracy check against a small gold set

You labeled a few by hand. Where does the AI disagree — and is *it* wrong, or are *you*? That
question is the whole skill.

In [ ]:
gold = ["positive", "mixed", "negative", "mixed", "positive"]  # your hand labels
df["gold"] = gold
disagree = df[df["label"] != df["gold"]]
print("agreement:", round((df["label"] == df["gold"]).mean(), 2))
print("disagreements:\n", disagree[["text","label","gold"]].to_string(index=False) if len(disagree) else "  none")

## You read your whole corpus at scale

You borrowed a powerful, opaque reader, made the prompt your codebook, used confidence to decide
what to trust, and checked it against your own judgment. The reckoning underneath: that judgment
was learned from creative work scraped largely without permission — whose reading are you
renting?

**Sketch:** show one label the AI got confidently wrong and one it nailed; for each, say how you
knew.